## Original SCB agent prompt provided in the paper
### Before the modular impl. agents are implemented, use this 
Specify additionally that follow the design in `current_design.json`

In [9]:
from prompts.scb import get_original_scb_prompt

# Pass the current checkpoint number to be implemented 
print(get_original_scb_prompt(1))


Implement a program that 100% solves the specification.
That is all you need to do.

Keep using the same virtual environment you started with,
update 'requirements.txt' with any new dependencies you need.

You are working on the following issue:
Issue path: checkpoint_2.md
Issue implementation path: checkpoint_2/
Extend your solution based on: checkpoint_1/




## Reader agent
Its purpose is to intiialise the `current_design.json` for the loop, and does not output anything. 

In [1]:
from prompts.reader import get_reader_prompt

print(get_reader_prompt(1))

This agent is not needed for the first checkpoint, please return.


## Analyzer agent

## IMPORTANT! TEST THIS AGAIN USING THE EXTRACTED / PROCESSED METRICS NOT THE PATH


In [2]:
from prompts.analyzer import get_analyzer_prompt

# The actual DPy results should be filtered for specific things we concern only. 
# For this TEST example, a path is provided instead. DO NOT do this in your final submission 
print(get_analyzer_prompt(1, second_iteration=True))


You are a senior software code quality analyst. 

Your job is to analyse the following modular design including kept, changed or new modules: 
Project root: agent_workspace
File: `current_design.json`
You are also given a list of previously suggested improvements that are rejected.

In addition, you are given the dependency graph in `current_deps_graph.json`, and a list of flagged code smells in `current_metrics/`. While they do not necessarily mean refactoring is needed, they may guide you in providing improvement suggestions. 

Output a JSON object decsribing improvement suggestions to the modules, and whether the design passes for implementation, using the following schema. If a module does not require refactoring, do not include it in the output. If no modules need refactoring, return an empty array. 
{
  "type": "object",
  "properties": {
    "result": {
      "description": "A flag indicating whether the design achieves a code quality suitable for implementation, ensuring long 

In [4]:
ANALYZER_OUTPUT = {
  "result": "fail",
  "improvements": [
    {
      "module_name": "pipeline.lexer",
      "smell": "The `tokenize` method is excessively long (139 lines) with a cyclomatic complexity of 38, containing a complex conditional with 4 conditions for number parsing.",
      "improvement_instruction": "Decompose `tokenize` into focused private helper methods, one per token category: `_scan_comment`, `_scan_dollar_expr`, `_scan_string`, `_scan_number`, `_scan_identifier`, `_scan_operator`. Each helper should handle its own token category and append to the token list, reducing `tokenize` to a dispatch loop."
    },
    {
      "module_name": "pipeline.parser",
      "smell": "`parse_task` is too long (87 lines, CC=23) and `parse_success_block` has high complexity (CC=10); additionally, `parse_pipeline_file` is flagged for feature envy as it also orchestrates lexer logic.",
      "improvement_instruction": "Split `parse_task` by extracting one private method per field handler: `_parse_run_field`, `_parse_success_field`, `_parse_requires_field`, `_parse_output_field`, `_parse_timeout_field`. Split `parse_success_block` into `_parse_success_inline_expr` and `_parse_success_braced_block`. Move the file-reading and lexer-invocation logic out of `parse_pipeline_file` so it only coordinates the `Parser` call, keeping lexing concerns inside `lexer`."
    },
    {
      "module_name": "pipeline.expr_parser",
      "smell": "`ExprParser` is insufficiently modularized (NOPM=22, WMC=85); `parse_for` has CC=17 with four complex conditionals; `parse_stmt` and `parse_primary` are both too long (67 and 84 lines respectively).",
      "improvement_instruction": "Break `parse_for` into three private helpers: `_parse_for_init`, `_parse_for_condition`, `_parse_for_update`, reducing its body to a coordination sequence. Split `parse_stmt` by extracting `_parse_decl_stmt` (for typed variable declarations) and `_parse_assign_stmt` (for assignments and increment/decrement). Split `parse_primary` by extracting `_parse_primary_literal`, `_parse_primary_dollar`, and `_parse_primary_ident` to handle each primary expression group."
    },
    {
      "module_name": "pipeline.evaluator",
      "smell": "`_exec_stmt` has a cyclomatic complexity of 26; multiple methods (`eval_block`, `_exec_stmt`, `_eval_func`, `_resolve_io_source`) contain empty catch blocks that silently swallow exceptions.",
      "improvement_instruction": "Replace all bare `except` or `except Exception: pass` blocks with explicit exception types and either re-raise, log, or return a meaningful sentinel value. Decompose `_exec_stmt` by extracting `_exec_if_stmt`, `_exec_for_stmt`, `_exec_while_stmt`, and `_exec_assign_stmt` as private methods, so `_exec_stmt` becomes a dispatch table with a single `isinstance` check per branch."
    },
    {
      "module_name": "pipeline.executor",
      "smell": "`Executor` has multiple responsibilities (multifaceted abstraction, LCOM=1): it handles shell execution, requires evaluation, success criterion evaluation, cycle detection, and job recording; `_exec_requires_stmt` has CC=18; `_evaluate_success_criterion` is flagged for feature envy (belongs in `evaluator`); `Executor` directly accesses private `Evaluator` members (`_eval`, `_exec_stmt`, `_truthy`); `JobContext.__init__` has 8 parameters; magic numbers `124` and `2` are used inline.",
      "improvement_instruction": "Move `_evaluate_success_criterion` into the `Evaluator` class and expose it as a public method, eliminating the cross-class private-member access. Replace the magic number `124` with a named constant `TIMEOUT_EXIT_CODE = 124` and `2` with a named constant `ESCAPE_STEP = 2`. Reduce `JobContext.__init__` to at most 5 parameters by grouping `workspace`, `cwd`, and `output_dir` into a `PathContext` dataclass. Decompose `_exec_requires_stmt` by extracting `_exec_requires_if`, `_exec_requires_for`, and `_exec_requires_while` helpers. Consider separating shell execution logic (`_execute_run`, `_build_shell_script`, `_process_string_escapes`) into a dedicated `ShellRunner` class to address the multifaceted abstraction smell."
    },
    {
      "module_name": "pipeline.main",
      "smell": "`main` is too long (70 lines, CC=15) and `_split_args` has CC=12 with a magic number `2`; two empty catch blocks silently discard `ValueError` in `_parse_literal`.",
      "improvement_instruction": "Extract config-loading logic from `main` into a `_load_config(path)` helper that returns an env dict and entry invocation string. Replace the two silent `except ValueError: pass` blocks in `_parse_literal` with explicit fall-through logic (try int, then float, then return the raw string) without exception swallowing. Replace the magic number `2` in `_split_args` with a named constant `ESCAPE_ADVANCE = 2`."
    },
    {
      "module_name": "pipeline",
      "smell": "Feature concentration: the package mixes five independent concern clusters (lexer/parser, evaluator, ast_nodes, executor/main, expr_parser) in a single flat package, yielding an LCC of 0.71.",
      "improvement_instruction": "Reorganise the package into sub-packages that align with the independent clusters identified: `pipeline/frontend/` for `lexer`, `parser`, and `ast_nodes`; `pipeline/runtime/` for `evaluator`, `expr_parser`, and `executor`; keep `pipeline/main.py` as the sole CLI entry point. Update all intra-package imports accordingly."
    }
  ]
}

## Decomposer Agent

In [1]:
from prompts.decomposer import get_decomposer_prompt

# Extract only the improvements part of the analyzer output 
print(get_decomposer_prompt(1))


You are a senior software engineer that specialises in modular software design.

You are working on the following issue:
Project root: agent_workspace
Issue path: checkpoint_1.md







Propose a modular design, each with a single responsibility, that when integrated together, achieve the goal specified in the issue.

Overwrite the JSON object in `current_design.json` by including all modules in your design, that can either be kept, changed or new, using the following schema.
{
  "type": "object",
  "properties": {
    "modules": {
      "description": "A list of modules to be implemented to solve the problem.",
      "type": "array",
      "items": {
        "oneOf": [
          {
            "type": "object",
            "description": "An existing module that remains unchanged.",
            "properties": {
              "type": {
                "type": "string",
                "const": "keep"
              },
              "module_name": {
                "description": "The nam

## Analyzer second time

In [6]:
from prompts.analyzer import get_analyzer_prompt

print(get_analyzer_prompt(2, True))


You are a senior software code quality analyst. 

Your job is to analyse the following modular design including kept, changed or new modules: 
Project root: agent_workspace
File: current_design.json
You are also given a list of previously suggested improvements that are rejected.

In addition, you are given the dependency graph in `current_deps_graph.json`, and a list of flagged code smells in `current_metrics/`. While they do not necessarily mean refactoring is needed, they may guide you in providing improvement suggestions. 

Output a JSON object decsribing improvement suggestions to the modules using the following schema. If a module does not require refactoring, do not include it in the output. If no modules need refactoring, return an empty array. 
{
  "type": "object",
  "properties": {
    "result": {
      "description": "A flag indicating whether the design achieves a code quality suitable for implementation, ensuring long term maintainability of the code as it evolves.",
   

In [7]:
ANALYZER_OUTPUT_2 = {
    "result": "pass",
    "improvements": [
      {
        "module_name": "pipeline.ast_nodes",
        "smell": "Module conflates language-level AST node types with application-level caching configuration structures, reducing cohesion and coupling the language frontend to the caching subsystem.",
        "improvement_instruction": "Extract TtlConfig, CacheKeyConfig, CacheConfig, and GlobalCacheConfig into a dedicated pipeline.cache_config module. pipeline.ast_nodes should retain only constructs that represent parsed language elements (TaskDef, ParamDef, and token-adjacent types). Update imports in pipeline.cache_key, pipeline.cache_manager, pipeline.executor, and pipeline.main accordingly."
      },
      {
        "module_name": "pipeline.expr_parser",
        "smell": "parse_block returns List[Any], erasing all type information at the parser/evaluator boundary despite typed AST node dataclasses existing in pipeline.ast_nodes.",
        "improvement_instruction": "Define a StmtNode union type or a common base dataclass in pipeline.ast_nodes that covers all statement node variants (IfStmt, ForStmt, WhileStmt, AssignStmt, ReturnStmt, etc.). Change ExprParser.parse_block to return List[StmtNode] so that static type checking is preserved across the parse/evaluate boundary."
      },
      {
        "module_name": "pipeline.evaluator",
        "smell": "eval_block accepts raw List[Token] and internally invokes ExprParser, conflating token parsing with expression evaluation and violating the established lex-parse-evaluate layering.",
        "improvement_instruction": "Remove the token-to-AST parsing step from eval_block. Change its signature to accept List[StmtNode] (a pre-parsed AST). Callers such as Executor should invoke ExprParser.parse_block explicitly before calling eval_block, keeping the two phases separately testable and aligned with the pipeline.expr_parser/pipeline.evaluator module boundary."
      },
      {
        "module_name": "pipeline.cache_manager",
        "smell": "store() accepts both the precomputed cache_key and the raw inputs (task_def, params, workspace) from which the key was derived, producing a redundant and inconsistent method signature.",
        "improvement_instruction": "Simplify store() to accept only task_def (for cache location resolution), cache_key, and job_result. Remove the redundant params and workspace parameters; since cache_key is already computed by a prior check() call, only the cache directory (derivable from task_def.cache.location) is needed to persist the entry."
      },
      {
        "module_name": "pipeline.cache_store",
        "smell": "The exists() method is fully subsumed by load() returning None, unnecessarily widening the public interface and enabling TOCTOU access patterns.",
        "improvement_instruction": "Remove the exists() method from cache_store's public interface. Update all callers in pipeline.cache_manager to use load() and branch on the None return value, eliminating the separate existence check."
      }
    ]
  }